### Check Python version

In [1]:
from platform import python_version_tuple

assert (
    python_version_tuple()[:2] == ("3", "11")
), "You are not running Python 3.11. Make sure to run Python through the course Conda environment."

In [2]:
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from tqdm import tqdm
import tifffile as tiff


# EXTRACTION OF IMAGES STARTS HERE

In [18]:
# =====================================================
# CONFIGURATION
# =====================================================

IMAGE_DIR = Path("potsdam/Ortho_RGB_val_and_test")
LABEL_DIR = Path("potsdam/Labels_all")


image_files = sorted(IMAGE_DIR.glob("*.tif"))

val_image_files = sorted(np.random.choice(image_files, size=5, replace=False))
test_image_files = sorted([f for f in image_files if f not in val_image_files])


print("Val image files:")
for f in val_image_files:
    print(f.name)

print("\nTest image files:")
for f in test_image_files:
    print(f.name)



Val image files:
top_potsdam_2_14_RGB.tif
top_potsdam_3_13_RGB.tif
top_potsdam_4_15_RGB.tif
top_potsdam_5_14_RGB.tif
top_potsdam_6_15_RGB.tif

Test image files:
top_potsdam_2_13_RGB.tif
top_potsdam_3_14_RGB.tif
top_potsdam_4_13_RGB.tif
top_potsdam_4_14_RGB.tif
top_potsdam_5_13_RGB.tif
top_potsdam_5_15_RGB.tif
top_potsdam_6_13_RGB.tif
top_potsdam_6_14_RGB.tif
top_potsdam_7_13_RGB.tif


## Extraction of Validation patches

- create patches of dimension (512 x 512 x 3)
- stride == 512 i.e. no overlapp
- also NO overlap for the rightmost or bottom part of the original image tile
    - these areas are not included to avoid predictions of the same areas twice

In [19]:

OUT_IMAGE_DIR = Path("val_patches/images")
OUT_LABEL_DIR = Path("val_patches/labels")

PATCH_SIZE = 512
STRIDE = 512  # 512 = no overlap

OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

In [20]:
# =====================================================
# EXTRACTION of Validation patches
# =====================================================


for image_path in tqdm(val_image_files, desc="Extracting validation patches"):

    # Labels have same name as image with ending "_RGB" replaced with "_label")
    label_path = LABEL_DIR / image_path.name.replace("_RGB", "_label")

    if not label_path.exists():
        print(f"Missing label for {image_path.name}")
        continue

    image = tiff.imread(image_path)
    label = tiff.imread(label_path)

    # shape of image read using tifffile is (H, W, C) 
    # for Potsdam images, this is (6000, 6000, 3) 
    h, w = image.shape[0], image.shape[1]


    stem = (image_path.stem).replace("_RGB", "")

    patch_count = 0

    # Calculate the x and y starting coordinates of the patches
    xs = list(range(0, w - PATCH_SIZE, STRIDE))
    ys = list(range(0, h - PATCH_SIZE, STRIDE))

    
    for y in ys:
        for x in xs:

            image_patch = image[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]

            label_patch = label[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]

            
            patch_name = f"{stem}_{y}_{x}.tif"

            tiff.imwrite(
                OUT_IMAGE_DIR / patch_name,
                image_patch )

            tiff.imwrite(
                OUT_LABEL_DIR / patch_name,
                label_patch )

            patch_count += 1

    print(f"From image tile {image_path.name}: {patch_count} patches extracted")

print("Finished.")

Extracting validation patches:  20%|██        | 1/5 [00:04<00:17,  4.43s/it]

From image tile top_potsdam_2_14_RGB.tif: 121 patches extracted


Extracting validation patches:  40%|████      | 2/5 [00:08<00:12,  4.19s/it]

From image tile top_potsdam_3_13_RGB.tif: 121 patches extracted


Extracting validation patches:  60%|██████    | 3/5 [00:12<00:08,  4.15s/it]

From image tile top_potsdam_4_15_RGB.tif: 121 patches extracted


Extracting validation patches:  80%|████████  | 4/5 [00:16<00:04,  4.17s/it]

From image tile top_potsdam_5_14_RGB.tif: 121 patches extracted


Extracting validation patches: 100%|██████████| 5/5 [00:20<00:00,  4.20s/it]

From image tile top_potsdam_6_15_RGB.tif: 121 patches extracted
Finished.


## Extraction of Test patches

- create patches of dimension (512 x 512 x 3)
- stride == 512 i.e. no overlapp
- also NO overlap for the rightmost or bottom part of the original image tile
    - these areas are not included to avoid predictions of the same areas twice

In [21]:

OUT_IMAGE_DIR = Path("test_patches/images")
OUT_LABEL_DIR = Path("test_patches/labels")

PATCH_SIZE = 512
STRIDE = 512  # 512 = no overlap

OUT_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
OUT_LABEL_DIR.mkdir(parents=True, exist_ok=True)

In [22]:
# =====================================================
# EXTRACTION of Validation patches
# =====================================================


for image_path in tqdm(test_image_files, desc="Extracting test patches"):

    # Labels have same name as image with ending "_RGB" replaced with "_label")
    label_path = LABEL_DIR / image_path.name.replace("_RGB", "_label")

    if not label_path.exists():
        print(f"Missing label for {image_path.name}")
        continue

    image = tiff.imread(image_path)
    label = tiff.imread(label_path)

    # shape of image read using tifffile is (H, W, C) 
    # for Potsdam images, this is (6000, 6000, 3) 
    h, w = image.shape[0], image.shape[1]


    stem = (image_path.stem).replace("_RGB", "")

    patch_count = 0

    # Calculate the x and y starting coordinates of the patches
    xs = list(range(0, w - PATCH_SIZE, STRIDE))
    ys = list(range(0, h - PATCH_SIZE, STRIDE))

    
    for y in ys:
        for x in xs:

            image_patch = image[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]

            label_patch = label[
                y:y + PATCH_SIZE,
                x:x + PATCH_SIZE ]

            
            patch_name = f"{stem}_{y}_{x}.tif"

            tiff.imwrite(
                OUT_IMAGE_DIR / patch_name,
                image_patch )

            tiff.imwrite(
                OUT_LABEL_DIR / patch_name,
                label_patch )

            patch_count += 1

    print(f"From image tile {image_path.name}: {patch_count} patches extracted")

print("Finished.")

Extracting test patches:  11%|█         | 1/9 [00:04<00:34,  4.32s/it]

From image tile top_potsdam_2_13_RGB.tif: 121 patches extracted


Extracting test patches:  22%|██▏       | 2/9 [00:08<00:29,  4.20s/it]

From image tile top_potsdam_3_14_RGB.tif: 121 patches extracted


Extracting test patches:  33%|███▎      | 3/9 [00:12<00:24,  4.12s/it]

...

From image tile top_potsdam_6_13_RGB.tif: 121 patches extracted


Extracting test patches:  89%|████████▉ | 8/9 [00:33<00:04,  4.32s/it]

From image tile top_potsdam_6_14_RGB.tif: 121 patches extracted


Extracting test patches: 100%|██████████| 9/9 [00:38<00:00,  4.24s/it]

From image tile top_potsdam_7_13_RGB.tif: 121 patches extracted
Finished.
